In [1]:
# Notebook 03: Tao dac trung va artifacts cho recommendation model

In [1]:
import pandas as pd
import numpy as np
import re
import ast
import json
import joblib
import time
from pathlib import Path
from datetime import datetime
from scipy import sparse
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


In [2]:
df_recipes = pd.read_csv("../data/processed/recipes_clean.csv")
df_interactions = pd.read_csv("../data/processed/interactions_clean.csv")

def parse_list_safe(x):
    if isinstance(x, list):
        return x
    if pd.isna(x):
        return []
    if isinstance(x, str):
        try:
            v = ast.literal_eval(x.strip())
            return v if isinstance(v, list) else []
        except (ValueError, SyntaxError):
            return []
    return []

for col in ["ingredients_clean", "tags_clean", "ingredients", "tags", "steps"]:
    if col in df_recipes.columns:
        df_recipes[col] = df_recipes[col].apply(parse_list_safe)

print(f"recipes:      {df_recipes.shape}")
print(f"interactions: {df_interactions.shape}")
print(f"Sample ingredients_clean: {df_recipes['ingredients_clean'].iloc[0][:3]}")
print(f"Sample tags_clean:        {df_recipes['tags_clean'].iloc[0][:3]}")


recipes:      (76608, 24)
interactions: (693880, 5)
Sample ingredients_clean: ['winter squash', 'mexican seasoning', 'mixed spice']
Sample tags_clean:        ['60-minutes-or-less', 'time-to-make', 'course']


In [3]:
# Kiểm tra kiểu dữ liệu, lọc orphan và đồng bộ lại sau khi lọc recipes

print(f"Truoc: recipes {df_recipes.shape}, interactions {df_interactions.shape}")

df_recipes["id"] = pd.to_numeric(df_recipes["id"], errors="coerce").astype("Int64")
df_recipes["minutes"] = pd.to_numeric(df_recipes["minutes"], errors="coerce")
df_recipes["calories"] = pd.to_numeric(df_recipes["calories"], errors="coerce")

df_interactions["user_id"] = pd.to_numeric(df_interactions["user_id"], errors="coerce").astype("Int64")
df_interactions["recipe_id"] = pd.to_numeric(df_interactions["recipe_id"], errors="coerce").astype("Int64")
df_interactions["rating"] = pd.to_numeric(df_interactions["rating"], errors="coerce")
if "date" in df_interactions.columns:
    df_interactions["date"] = pd.to_datetime(df_interactions["date"], errors="coerce")

valid_ids = set(df_recipes["id"].dropna())
n_before = len(df_interactions)
df_interactions = df_interactions[df_interactions["recipe_id"].isin(valid_ids)].copy()
print(f"Bo {n_before - len(df_interactions):,} dong tuong tac recipe_id khong co trong recipes")

df_recipes = df_recipes[df_recipes["minutes"].between(1, 1440)].copy()
df_recipes["is_calorie_outlier"] = ~df_recipes["calories"].between(10, 5000)
print(f"Mon co calories ngoai 10-5000: {df_recipes['is_calorie_outlier'].sum():,} (van giu, co cot danh dau)")

df_interactions = df_interactions[df_interactions["recipe_id"].isin(set(df_recipes["id"]))].copy()

print(f"Sau: recipes {df_recipes.shape}, interactions {df_interactions.shape}")
print(f"User: {df_interactions['user_id'].nunique():,}, recipe trong interactions: {df_interactions['recipe_id'].nunique():,}")

Truoc: recipes (76608, 24), interactions (693880, 5)
Bo 0 dong tuong tac recipe_id khong co trong recipes
Mon co calories ngoai 10-5000: 0 (van giu, co cot danh dau)
Sau: recipes (76608, 25), interactions (693880, 5)
User: 30,836, recipe trong interactions: 76,608


In [4]:
# Tach train/val theo thoi gian (80% cu nhat -> train, 20% moi nhat -> val)

interactions_sorted = df_interactions.sort_values("date").reset_index(drop=True)
split_idx = int(len(interactions_sorted) * 0.8)

train_interactions = interactions_sorted.iloc[:split_idx].copy()
val_interactions = interactions_sorted.iloc[split_idx:].copy()

train_user_ids = set(train_interactions["user_id"])
train_recipe_ids = set(train_interactions["recipe_id"])
val_user_ids = set(val_interactions["user_id"])
val_recipe_ids = set(val_interactions["recipe_id"])

cold_start_users = val_user_ids - train_user_ids
cold_start_items = val_recipe_ids - train_recipe_ids

print(f"Train: {len(train_interactions):,} dong | Val: {len(val_interactions):,} dong")
print(f"Train date: {train_interactions['date'].min()} -> {train_interactions['date'].max()}")
print(f"Val date:   {val_interactions['date'].min()} -> {val_interactions['date'].max()}")
print(f"User train/val: {len(train_user_ids):,} / {len(val_user_ids):,} | Recipe train/val: {len(train_recipe_ids):,} / {len(val_recipe_ids):,}")
print(f"User cold-start trong val: {len(cold_start_users):,} | Recipe cold-start trong val: {len(cold_start_items):,}")

Train: 555,104 dong | Val: 138,776 dong
Train date: 2000-02-25 00:00:00 -> 2010-08-14 00:00:00
Val date:   2010-08-14 00:00:00 -> 2018-12-19 00:00:00
User train/val: 27,016 / 14,353 | Recipe train/val: 72,213 / 43,731
User cold-start trong val: 3,820 | Recipe cold-start trong val: 4,395


In [5]:
# Tao cot text cho TF-IDF (uu tien ingredients va tags)

def normalize_text(text):
    if pd.isna(text):
        return ""
    text = str(text).lower()
    text = re.sub(r"[^\w\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def normalize_ingredient(token):
    if pd.isna(token):
        return ""
    token = str(token).lower().strip().replace(" ", "_")
    return token


def list_to_text(lst):
    if not isinstance(lst, list):
        return ""
    return " ".join(normalize_ingredient(x) for x in lst)


def tags_to_text(lst):
    if not isinstance(lst, list):
        return ""
    return " ".join(normalize_text(x) for x in lst)


df_recipes["ingredients_text"] = df_recipes["ingredients_clean"].apply(list_to_text)
df_recipes["tags_text"] = df_recipes["tags_clean"].apply(tags_to_text)

if "description" in df_recipes.columns:
    df_recipes["description_clean"] = df_recipes["description"].apply(normalize_text)
else:
    df_recipes["description_clean"] = ""


def build_combined_text(row):
    ing, tags, desc = row["ingredients_text"], row["tags_text"], row["description_clean"]
    return " ".join([ing, ing, tags, tags, desc]).strip()


df_recipes["combined_text"] = df_recipes.apply(build_combined_text, axis=1)

display(df_recipes[["id", "ingredients_text", "tags_text", "description_clean", "combined_text"]].head())
print("Null:", df_recipes[["ingredients_text", "tags_text", "combined_text"]].isnull().sum().to_dict())
print("Do dai trung binh combined_text:", round(df_recipes["combined_text"].str.len().mean(), 1))

,id,ingredients_text,tags_text,description_clean,combined_text
0,137739,winter_squash mexican_seasoning mixed_spice ho...,60 minutes or less time to make course main in...,autumn is my favorite time of year to cook thi...,winter_squash mexican_seasoning mixed_spice ho...
1,75452,sugar unsalted_butter bananas eggs fresh_lemon...,weeknight time to make course main ingredient ...,from ann hodgman s,sugar unsalted_butter bananas eggs fresh_lemon...
2,63986,lean_pork_chops flour salt dry_mustard garlic_...,weeknight time to make course main ingredient ...,here s and old standby i enjoy from time to ti...,lean_pork_chops flour salt dry_mustard garlic_...
3,43026,egg_roll_wrap whole_green_chilies cheese corns...,60 minutes or less time to make course main in...,a favorite from a local restaurant no longer i...,egg_roll_wrap whole_green_chilies cheese corns...
4,23933,butterscotch_chips chinese_noodles salted_peanuts,15 minutes or less time to make course prepara...,a little different and oh so good i include th...,butterscotch_chips chinese_noodles salted_pean...


Null: {'ingredients_text': 0, 'tags_text': 0, 'combined_text': 0}
Do dai trung binh combined_text: 844.9


In [6]:
# TF-IDF cho combined_text -> vector thua cho moi recipe

tfidf_vectorizer = TfidfVectorizer(
    max_features=20000,
    ngram_range=(1, 2),
    min_df=5,
    max_df=0.95,
    sublinear_tf=True,
)

recipe_tfidf_matrix = tfidf_vectorizer.fit_transform(df_recipes["combined_text"])

recipe_index_map = pd.DataFrame({
    "recipe_id": df_recipes["id"].values,
    "matrix_index": range(len(df_recipes)),
})

print("Matrix:", recipe_tfidf_matrix.shape, "| vocab:", len(tfidf_vectorizer.vocabulary_))
sparsity = 1 - recipe_tfidf_matrix.nnz / (recipe_tfidf_matrix.shape[0] * recipe_tfidf_matrix.shape[1])
print("Sparsity:", f"{sparsity:.2%}")

display(recipe_index_map.head())

Matrix: (76608, 20000) | vocab: 20000
Sparsity: 99.47%


,recipe_id,matrix_index
0,137739,0
1,75452,1
2,63986,2
3,43026,3
4,23933,4


In [7]:
# Top-K recipe tuong tu (cosine) tu TF-IDF, tinh theo batch de tiet RAM

TOP_K = 50
BATCH_SIZE = 500

n_recipes = recipe_tfidf_matrix.shape[0]
all_recipe_ids = recipe_index_map["recipe_id"].values

t0 = time.time()
results = []

for start in range(0, n_recipes, BATCH_SIZE):
    end = min(start + BATCH_SIZE, n_recipes)
    sim_matrix = cosine_similarity(
        recipe_tfidf_matrix[start:end],
        recipe_tfidf_matrix,
    )

    for local_idx in range(sim_matrix.shape[0]):
        global_idx = start + local_idx
        scores = sim_matrix[local_idx]
        scores[global_idx] = -1.0

        top_k = np.argpartition(scores, -TOP_K)[-TOP_K:]
        top_k = top_k[np.argsort(scores[top_k])[::-1]]

        src_id = all_recipe_ids[global_idx]
        for j in top_k:
            results.append((src_id, all_recipe_ids[j], float(scores[j])))

    if end % 5000 == 0 or end == n_recipes:
        print(f"{end:,}/{n_recipes:,} | {time.time() - t0:.1f}s")

recipe_similarity_topk = pd.DataFrame(
    results,
    columns=["recipe_id", "neighbor_recipe_id", "similarity"],
)

print("Xong:", round(time.time() - t0, 1), "s | shape:", recipe_similarity_topk.shape)

sample_id = all_recipe_ids[0]
nb = recipe_similarity_topk[recipe_similarity_topk["recipe_id"] == sample_id].head(5)
print("Vi du 5 neighbor dau tien cua recipe_id =", sample_id)
for _, row in nb.iterrows():
    name = df_recipes.loc[df_recipes["id"] == row["neighbor_recipe_id"], "name"]
    name = name.iloc[0] if len(name) else "?"
    print(int(row["neighbor_recipe_id"]), round(row["similarity"], 4), name)

5,000/76,608 | 15.7s
10,000/76,608 | 28.5s
15,000/76,608 | 42.2s
20,000/76,608 | 55.5s
25,000/76,608 | 68.8s
30,000/76,608 | 82.1s
35,000/76,608 | 95.3s
40,000/76,608 | 109.2s
45,000/76,608 | 122.6s
50,000/76,608 | 135.6s
55,000/76,608 | 148.6s
60,000/76,608 | 162.0s
65,000/76,608 | 176.1s
70,000/76,608 | 189.6s
75,000/76,608 | 202.9s
76,608/76,608 | 207.2s
Xong: 210.3 s | shape: (3830400, 3)
Vi du 5 neighbor dau tien cua recipe_id = 137739
235285 0.2889 mexican squash
216993 0.2869 sweet and spicy yams
189352 0.2823 maple sweet dumpling squash
267785 0.2779 parmesan butternut squash gratin
96811 0.2626 zucotte


In [8]:
# Cot so va nhan (thoi gian, calories, do phuc tap)

df_recipes["is_quick_meal"] = df_recipes["minutes"] <= 30
df_recipes["is_medium_meal"] = df_recipes["minutes"].between(31, 60)
df_recipes["is_long_meal"] = df_recipes["minutes"] > 60

df_recipes["is_low_calorie"] = df_recipes["calories"].between(10, 300)
df_recipes["is_medium_calorie"] = df_recipes["calories"].between(301, 600)
df_recipes["is_high_calorie"] = df_recipes["calories"] > 600

df_recipes["is_simple_recipe"] = df_recipes["n_ingredients_calc"] <= 5
df_recipes["is_complex_recipe"] = df_recipes["n_ingredients_calc"] >= 15

recipe_features = df_recipes[[
    "id", "minutes", "calories", "n_ingredients_calc",
    "is_calorie_outlier",
    "is_quick_meal", "is_medium_meal", "is_long_meal",
    "is_low_calorie", "is_medium_calorie", "is_high_calorie",
    "is_simple_recipe", "is_complex_recipe",
]].copy().rename(columns={"id": "recipe_id"})

print("recipe_features:", recipe_features.shape)
display(recipe_features.head())

recipe_features: (76608, 13)


,recipe_id,minutes,calories,n_ingredients_calc,is_calorie_outlier,is_quick_meal,is_medium_meal,is_long_meal,is_low_calorie,is_medium_calorie,is_high_calorie,is_simple_recipe,is_complex_recipe
0,137739,55,51.5,7,False,False,True,False,True,False,False,False,False
1,75452,70,2669.3,9,False,False,False,True,False,False,True,False,False
2,63986,500,105.7,7,False,False,False,True,True,False,False,False,False
3,43026,45,94.0,5,False,False,True,False,True,False,False,True,False
4,23933,15,232.7,3,False,True,False,False,True,False,False,True,False


In [9]:
# User features chi tu train_interactions

user_stats = train_interactions.groupby("user_id").agg(
    avg_rating_given=("rating", "mean"),
    rating_count=("rating", "count"),
    min_date=("date", "min"),
    max_date=("date", "max"),
).reset_index()

user_stats["activity_level"] = pd.cut(
    user_stats["rating_count"],
    bins=[0, 5, 20, np.inf],
    labels=["cold", "warm", "active"],
)

high_rated = train_interactions[train_interactions["rating"] >= 4][["user_id", "recipe_id"]]
high_rated = high_rated.merge(
    df_recipes[["id", "tags_clean"]].rename(columns={"id": "recipe_id"}),
    on="recipe_id",
    how="left",
)

def get_top_tags(tags_series, top_n=3):
    from collections import Counter
    all_tags = []
    for tag_list in tags_series:
        if isinstance(tag_list, list):
            all_tags.extend(tag_list)
    if not all_tags:
        return []
    return [t for t, _ in Counter(all_tags).most_common(top_n)]

favorite_tags = high_rated.groupby("user_id")["tags_clean"].apply(get_top_tags).reset_index()
favorite_tags.columns = ["user_id", "favorite_tags"]

user_features = user_stats.merge(favorite_tags, on="user_id", how="left")
user_features["favorite_tags"] = user_features["favorite_tags"].apply(
    lambda x: x if isinstance(x, list) else []
)

global_avg_rating = train_interactions["rating"].mean()
global_top_tags = get_top_tags(df_recipes["tags_clean"], top_n=3)

print("user_features:", user_features.shape)
print("activity_level:", user_features["activity_level"].value_counts().to_dict())
print("fallback:", round(global_avg_rating, 2), global_top_tags)
display(user_features.head())

user_features: (27016, 7)
activity_level: {'cold': 13561, 'warm': 8988, 'active': 4467}
fallback: 4.71 ['preparation', 'time-to-make', 'course']


,user_id,avg_rating_given,rating_count,min_date,max_date,activity_level,favorite_tags
0,1533,4.847222,72,2002-02-19,2008-03-01,active,"[time-to-make, preparation, course]"
1,1535,4.569811,530,2004-05-22,2010-08-14,active,"[preparation, time-to-make, course]"
2,1634,4.323529,34,2001-07-05,2010-02-05,active,"[time-to-make, preparation, dietary]"
3,1676,4.500000,16,2002-07-24,2010-04-23,warm,"[main-ingredient, preparation, time-to-make]"
4,1773,4.000000,3,2000-03-13,2001-01-29,cold,"[weeknight, time-to-make, course]"


In [ ]:
# Popularity tu train: dem rating, trung binh, score Bayesian, cold item

recipe_stats = (
    train_interactions.groupby("recipe_id")
    .agg(rating_count=("rating", "count"), rating_mean=("rating", "mean"))
    .reset_index()
)

popularity_features = (
    df_recipes[["id"]]
    .rename(columns={"id": "recipe_id"})
    .merge(recipe_stats, on="recipe_id", how="left")
)

popularity_features["rating_count"] = popularity_features["rating_count"].fillna(0)
popularity_features["rating_mean"] = popularity_features["rating_mean"].fillna(0)

C = train_interactions["rating"].mean()
m = recipe_stats["rating_count"].quantile(0.75)

v = popularity_features["rating_count"]
R = popularity_features["rating_mean"]
popularity_features["popularity_score"] = (v / (v + m)) * R + (m / (v + m)) * C
popularity_features["is_cold_item"] = popularity_features["rating_count"] == 0

popularity_features = popularity_features[
    ["recipe_id", "rating_count", "rating_mean", "popularity_score", "is_cold_item"]
]

print("C =", round(C, 3), "| m =", round(m, 2))
print("shape:", popularity_features.shape, "| cold:", int(popularity_features["is_cold_item"].sum()))
display(popularity_features.sort_values("popularity_score", ascending=False).head())


C = 4.713 | m = 7.0
shape: (76608, 5) | cold: 4395


,recipe_id,rating_count,rating_mean,popularity_score,is_cold_item
377,77497,58.0,5.000000,4.969136,False
6477,34753,54.0,5.000000,4.967112,False
39607,154351,51.0,5.000000,4.965411,False
50875,78579,165.0,4.975758,4.965080,False
69997,186029,49.0,5.000000,4.964175,False


7.0


In [12]:
# Kiem tra nhanh truoc khi luu artifacts

def check_nulls(name, df):
    bad = df.isnull().sum()
    bad = bad[bad > 0]
    if len(bad):
        print(name, "null:", bad.to_dict())

check_nulls("recipe_features", recipe_features)
check_nulls("user_features", user_features)
check_nulls("popularity_features", popularity_features)

assert recipe_features["recipe_id"].is_unique, "recipe_id trung trong recipe_features"
assert user_features["user_id"].is_unique, "user_id trung trong user_features"
assert popularity_features["recipe_id"].is_unique, "recipe_id trung trong popularity_features"

n_recipes = df_recipes["id"].nunique()
n_users_train = train_interactions["user_id"].nunique()

assert len(recipe_features) == n_recipes, "recipe_features khong du so mon"
assert len(popularity_features) == n_recipes, "popularity_features khong du so mon"
assert len(user_features) == n_users_train, "user_features khong du so user train"

assert recipe_tfidf_matrix.shape[0] == n_recipes, "TF-IDF khong khop so hang voi recipes"
assert recipe_index_map["recipe_id"].is_unique, "recipe_index_map bi trung recipe_id"
assert len(recipe_index_map) == n_recipes, "recipe_index_map khong du dong"

TOPK_NEIGHBORS = 50
cnt = recipe_similarity_topk.groupby("recipe_id").size()
assert int(cnt.min()) == TOPK_NEIGHBORS, "Thieu neighbor cho mot so recipe"

ps = popularity_features["popularity_score"]
assert ps.min() >= 0 and ps.max() <= 5.5, "popularity_score ngoai khoang hop ly"

print("OK | recipes:", n_recipes, "| users train:", n_users_train)
print("TF-IDF:", recipe_tfidf_matrix.shape, "| similarity rows:", len(recipe_similarity_topk))

OK | recipes: 76608 | users train: 27016
TF-IDF: (76608, 20000) | similarity rows: 3830400


In [13]:
# Luu artifacts + manifest

OUTPUT_DIR = Path("../artifacts/features")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

for df_fix, cols in [
    (recipe_index_map, ["recipe_id"]),
    (recipe_features, ["recipe_id"]),
    (recipe_similarity_topk, ["recipe_id", "neighbor_recipe_id"]),
    (popularity_features, ["recipe_id"]),
    (user_features, ["user_id"]),
    (train_interactions, ["user_id", "recipe_id"]),
    (val_interactions, ["user_id", "recipe_id"]),
]:
    for c in cols:
        if c in df_fix.columns:
            df_fix[c] = df_fix[c].astype("int64")

def save_df(df, path, index=False):
    try:
        out = path.with_suffix(".parquet")
        df.to_parquet(out, index=index)
        return out
    except Exception:
        out = path.with_suffix(".csv")
        df.to_csv(out, index=index)
        return out

joblib.dump(tfidf_vectorizer, OUTPUT_DIR / "tfidf_vectorizer.joblib")
sparse.save_npz(OUTPUT_DIR / "recipe_tfidf_matrix.npz", recipe_tfidf_matrix)

save_df(recipe_index_map, OUTPUT_DIR / "recipe_index_map")
save_df(recipe_similarity_topk, OUTPUT_DIR / "recipe_similarity_topk")
save_df(recipe_features, OUTPUT_DIR / "recipe_features")

user_features_out = user_features.copy()
user_features_out["favorite_tags"] = user_features_out["favorite_tags"].apply(str)
save_df(user_features_out, OUTPUT_DIR / "user_features")

save_df(popularity_features, OUTPUT_DIR / "popularity_features")
save_df(train_interactions, OUTPUT_DIR / "train_interactions")
save_df(val_interactions, OUTPUT_DIR / "val_interactions")

saved_files = sorted(
    f.name for f in OUTPUT_DIR.iterdir()
    if f.is_file() and f.name != "feature_manifest.json"
)

manifest = {
    "version": "1.0",
    "build_timestamp": datetime.now().isoformat(),
    "source_data": {
        "recipes_clean": str(Path("../data/processed/recipes_clean.csv").resolve()),
        "interactions_clean": str(Path("../data/processed/interactions_clean.csv").resolve()),
    },
    "stats": {
        "n_recipes": int(len(recipe_features)),
        "n_users_train": int(len(user_features)),
        "n_train_interactions": int(len(train_interactions)),
        "n_val_interactions": int(len(val_interactions)),
        "tfidf_shape": list(recipe_tfidf_matrix.shape),
        "similarity_topk": int(recipe_similarity_topk.groupby("recipe_id").size().iloc[0]),
        "similarity_rows": int(len(recipe_similarity_topk)),
    },
    "config": {
        "tfidf_max_features": 20000,
        "tfidf_ngram_range": [1, 2],
        "tfidf_min_df": 5,
        "tfidf_max_df": 0.95,
        "similarity_top_k": 50,
        "train_val_split_ratio": 0.8,
        "split_method": "time-based",
    },
    "artifacts": saved_files,
}

with open(OUTPUT_DIR / "feature_manifest.json", "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2, ensure_ascii=False)

print("Da luu vao:", OUTPUT_DIR.resolve())
for name in saved_files + ["feature_manifest.json"]:
    p = OUTPUT_DIR / name
    print(name, round(p.stat().st_size / 1024 / 1024, 2), "MB")

Da luu vao: D:\GITHUB\Food-RecommendationSystem\artifacts\features
popularity_features.csv 3.24 MB
popularity_features.parquet 0.78 MB
recipe_features.csv 5.22 MB
recipe_features.parquet 0.9 MB
recipe_index_map.csv 1.0 MB
recipe_index_map.parquet 0.94 MB
recipe_similarity_topk.csv 124.28 MB
recipe_similarity_topk.parquet 39.77 MB
recipe_tfidf_matrix.npz 69.84 MB
tfidf_vectorizer.joblib 0.8 MB
train_interactions.csv 164.66 MB
train_interactions.parquet 91.51 MB
user_features.csv 2.39 MB
user_features.parquet 0.39 MB
val_interactions.csv 42.54 MB
val_interactions.parquet 23.57 MB
feature_manifest.json 0.0 MB
